# 05 – Modules & Packages

Topics covered:
- What is a module?
- `import` styles
- Creating your own module
- Packages and `__init__.py`
- The `__name__ == '__main__'` guard
- Key standard library modules: `os`, `sys`, `math`, `datetime`, `json`, `re`, `collections`, `itertools`, `pathlib`
- Virtual environments and package installation recap

## 1. What is a Module?

A **module** is any `.py` file. It contains functions, classes, and variables that can be reused across your project.

A **package** is a directory of modules with an `__init__.py` file.

## 1b. From-scratch: what `import` actually does

`import` is not magic. Conceptually, for a plain `.py` file, it does roughly four things:

1. **Locate** the file by searching a list of directories (`sys.path`).
2. **Read** its source text.
3. **Execute** that text in a fresh, isolated namespace — a dict, not the caller's own globals.
4. **Cache** the resulting namespace in `sys.modules`, keyed by module name, so a second `import`
   of the same module reuses the cached namespace instead of re-running the file.

The cell below builds a tiny, deliberately incomplete version of steps 2–3 by hand — reading a
`.py` file's text and `exec()`-ing it into a fresh namespace dict — then compares it against the
real mechanism (`importlib`) on the same file, to show what the `import` statement automates.

In [1]:
module_path = "/tmp/mini_greet_module.py"
with open(module_path, "w") as f:
    f.write(
        "GREETING = 'hello from mini module'\n"
        "\n"
        "def greet(name):\n"
        "    return f'{GREETING}, {name}!'\n"
    )


def manual_import(path):
    """A tiny, incomplete stand-in for what `import` automates."""
    source = open(path).read()                        # 1. read the file's text
    namespace = {"__name__": path, "__file__": path}   # 2. fresh, isolated namespace dict
    exec(compile(source, path, "exec"), namespace)     # 3. run the code IN that namespace
    return namespace                                   # 4. roughly what sys.modules[name] holds


mini = manual_import(module_path)
print("manual import  ->", mini["GREETING"])
print("manual import  ->", mini["greet"]("Ada"))

# The real mechanism, via importlib, on the identical file:
import importlib.util
spec = importlib.util.spec_from_file_location("mini_greet_module", module_path)
real_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(real_module)
print("real importlib ->", real_module.GREETING)
print("real importlib ->", real_module.greet("Ada"))

manual import  -> hello from mini module
manual import  -> hello from mini module, Ada!
real importlib -> hello from mini module
real importlib -> hello from mini module, Ada!


**Namespace isolation** is the part that matters most: each `manual_import` call gets its own
fresh dict, so two files that both define a variable called `X` never collide — unlike executing
both into one shared namespace, which silently overwrites the first with the second.

In [2]:
# Two files that both define a variable called `X` with different values.
with open("/tmp/mod_a.py", "w") as f:
    f.write("X = 'from module A'\n")
with open("/tmp/mod_b.py", "w") as f:
    f.write("X = 'from module B'\n")

ns_a = manual_import("/tmp/mod_a.py")
ns_b = manual_import("/tmp/mod_b.py")
print("ns_a['X'] =", ns_a["X"])
print("ns_b['X'] =", ns_b["X"])
print("same dict object?", ns_a is ns_b)

# Contrast: exec-ing both into ONE shared namespace collides.
shared = {}
exec(open("/tmp/mod_a.py").read(), shared)
exec(open("/tmp/mod_b.py").read(), shared)
print("shared['X'] after both =", shared["X"], "(module B silently overwrote module A's X)")

ns_a['X'] = from module A
ns_b['X'] = from module B
same dict object? False
shared['X'] after both = from module B (module B silently overwrote module A's X)


**Where step 1 (locate) and step 4 (cache) come from.** `sys.path` is the list of directories the
real `import` statement searches, in order, and `sys.modules` is the cache keyed by module name —
a module's top-level code runs exactly once, at first import; every later `import` of the same
name just returns the cached namespace.

In [3]:
import sys

print("first 3 entries of sys.path:")
for p in sys.path[:3]:
    print(" ", repr(p))

with open("/tmp/mod_c.py", "w") as f:
    f.write("print('mod_c is executing (top-level code runs once, at first import)')\nY = 42\n")

sys.path.insert(0, "/tmp")
import mod_c
print("first import:  mod_c.Y =", mod_c.Y)
import mod_c  # second import — no "is executing" print this time, it's cached
print("second import: mod_c.Y =", mod_c.Y)
print("cached in sys.modules?", "mod_c" in sys.modules)
sys.path.remove("/tmp")

first 3 entries of sys.path:
  '/home/yashwanth-aravind/.local/share/uv/python/cpython-3.13.9-linux-x86_64-gnu/lib/python313.zip'
  '/home/yashwanth-aravind/.local/share/uv/python/cpython-3.13.9-linux-x86_64-gnu/lib/python3.13'
  '/home/yashwanth-aravind/.local/share/uv/python/cpython-3.13.9-linux-x86_64-gnu/lib/python3.13/lib-dynload'
mod_c is executing (top-level code runs once, at first import)
first import:  mod_c.Y = 42
second import: mod_c.Y = 42
cached in sys.modules? True


In [4]:
# Import styles
import math                     # import entire module
import os as operating_system   # alias
from math import pi, sqrt       # import specific names
from math import *              # import everything (avoid — pollutes namespace)

print(math.pi)                  # 3.14159...
print(pi)                       # 3.14159... (imported directly)
print(sqrt(16))                 # 4.0
print(operating_system.getcwd())  # current working directory

3.141592653589793
3.141592653589793
4.0
/home/yashwanth-aravind/ml-course/python-bootcamp/01-python-foundation/05-modules-packages


## 2. `__name__ == '__main__'` Guard

When Python runs a file directly, `__name__` is `'__main__'`.  
When it's imported, `__name__` is the module name.  
This guard lets a file serve as both a script and an importable module.

In [5]:
# Simulating how this looks in a .py file:
def add(a, b):
    return a + b

# This block runs ONLY when this file is executed directly
if __name__ == '__main__':
    print("Running as a script")
    print(add(2, 3))
else:
    print(f"Imported as module: {__name__}")

Running as a script
5


## 3. Standard Library — `os` and `pathlib`

`pathlib.Path` is the modern way to work with file system paths.

In [6]:
import os
from pathlib import Path

# Current directory
print(os.getcwd())
print(Path.cwd())

# Path manipulation
p = Path(".") / "data" / "sample.csv"
print(p)              # data/sample.csv
print(p.parent)       # data
print(p.name)         # sample.csv
print(p.stem)         # sample
print(p.suffix)       # .csv
print(p.exists())     # True/False

/home/yashwanth-aravind/ml-course/python-bootcamp/01-python-foundation/05-modules-packages
/home/yashwanth-aravind/ml-course/python-bootcamp/01-python-foundation/05-modules-packages
data/sample.csv
data
sample.csv
sample
.csv
False


In [7]:
# Useful os operations
print(os.environ.get("HOME", "not set"))  # environment variables
print(os.path.join("dir", "subdir", "file.txt"))  # path join (old style)

# List directory contents
for entry in os.scandir("."):
    if not entry.name.startswith("."):
        print(entry.name, "(dir)" if entry.is_dir() else "")

/home/yashwanth-aravind
dir/subdir/file.txt
modules_packages.ipynb 
README.md 


## 4. Standard Library — `math` and `random`

In [8]:
import math
import random

# math
print(math.ceil(2.3))      # 3
print(math.floor(2.9))     # 2
print(math.log(100, 10))   # 2.0  (log base 10)
print(math.log2(8))        # 3.0
print(math.exp(1))         # e ≈ 2.718
print(math.factorial(5))   # 120
print(math.gcd(48, 18))    # 6
print(math.isclose(0.1 + 0.2, 0.3))  # True — safe float comparison

# random
random.seed(42)             # reproducibility
print(random.randint(1, 10))     # random int in [1, 10]
print(random.random())           # float in [0.0, 1.0)
print(random.choice(["a", "b", "c"]))  # random element
lst = [1, 2, 3, 4, 5]
random.shuffle(lst)
print(lst)
print(random.sample(lst, 3))     # 3 unique picks without replacement

3
2
2.0
3.0
2.718281828459045
120
6
True
2
0.025010755222666936
b
[4, 3, 1, 5, 2]
[2, 4, 1]


## 5. Standard Library — `datetime`

In [9]:
from datetime import date, time, datetime, timedelta

today = date.today()
now   = datetime.now()

print(today)              # 2025-05-05
print(now)                # 2025-05-05 14:23:01.123456
print(now.strftime("%d/%m/%Y %H:%M"))  # formatted string

# Parsing a string into a datetime
dt = datetime.strptime("2025-01-15", "%Y-%m-%d")
print(dt)

# Arithmetic
future = today + timedelta(days=30)
print(f"30 days from now: {future}")

diff = datetime(2025, 12, 31) - now
print(f"Days until end of year: {diff.days}")

2026-08-24
2026-08-24 01:53:49.949603
24/08/2026 01:53
2025-01-15 00:00:00
30 days from now: 2026-09-23
Days until end of year: -237


## 6. Standard Library — `json`

In [10]:
import json

# Python dict → JSON string
data = {"name": "Alice", "age": 30, "scores": [95, 87, 92]}
json_str = json.dumps(data, indent=2)
print(json_str)

# JSON string → Python dict
parsed = json.loads(json_str)
print(type(parsed), parsed["name"])

# Write to file
with open("/tmp/test_data.json", "w") as f:
    json.dump(data, f, indent=2)

# Read from file
with open("/tmp/test_data.json") as f:
    loaded = json.load(f)
print(loaded)

{
  "name": "Alice",
  "age": 30,
  "scores": [
    95,
    87,
    92
  ]
}
<class 'dict'> Alice
{'name': 'Alice', 'age': 30, 'scores': [95, 87, 92]}


## 7. Standard Library — `re` (Regular Expressions)

In [11]:
import re

text = "Contact us at support@example.com or sales@company.org"

# Find all email addresses
pattern = r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}"
emails = re.findall(pattern, text)
print(emails)

# Check if string matches pattern
is_email = bool(re.match(r"^\S+@\S+\.\S+$", "test@example.com"))
print(is_email)  # True

# Substitute
cleaned = re.sub(r"\s+", " ", "hello   world    python")
print(cleaned)  # hello world python

# Groups
match = re.search(r"(\d{4})-(\d{2})-(\d{2})", "Date: 2025-05-01")
if match:
    year, month, day = match.groups()
    print(f"Year: {year}, Month: {month}, Day: {day}")

['support@example.com', 'sales@company.org']
True
hello world python
Year: 2025, Month: 05, Day: 01


## 8. Standard Library — `collections` and `itertools`

In [12]:
from collections import Counter, defaultdict, OrderedDict, deque

# Counter — word frequency
words = "the quick brown fox jumps over the lazy dog the".split()
c = Counter(words)
print(c.most_common(3))

# deque — efficient append/pop from both ends
d = deque([1, 2, 3])
d.appendleft(0)
d.append(4)
print(d)
print(d.popleft(), d.pop())

[('the', 3), ('quick', 1), ('brown', 1)]
deque([0, 1, 2, 3, 4])
0 4


In [13]:
import itertools

# chain — combine iterables
combined = list(itertools.chain([1, 2], [3, 4], [5, 6]))
print(combined)

# product — Cartesian product
colors = ["red", "blue"]
sizes  = ["S", "M", "L"]
for combo in itertools.product(colors, sizes):
    print(combo, end=" ")
print()

# combinations and permutations
print(list(itertools.combinations("ABCD", 2)))
print(list(itertools.permutations("ABC", 2)))

[1, 2, 3, 4, 5, 6]
('red', 'S') ('red', 'M') ('red', 'L') ('blue', 'S') ('blue', 'M') ('blue', 'L') 
[('A', 'B'), ('A', 'C'), ('A', 'D'), ('B', 'C'), ('B', 'D'), ('C', 'D')]
[('A', 'B'), ('A', 'C'), ('B', 'A'), ('B', 'C'), ('C', 'A'), ('C', 'B')]


## 8b. Failure modes, reproduced for real

Two failures that follow directly from how `import`/`sys.path`/`sys.modules` actually work
(section 1b above), not arbitrary gotchas.

**Circular imports.** If `circ_a.py` does `from circ_b import Y` and `circ_b.py` does
`from circ_a import X`, importing `circ_a` first starts executing it, hits the `from circ_b
import Y` line, and starts executing `circ_b` — which immediately tries `from circ_a import X`.
`circ_a` is already in `sys.modules` (partially initialized — mid-execution, at the very first
line), but its `X = 1` line hasn't run yet, so the name isn't there yet. Python raises
`ImportError`, not silently returning a stale or empty value.

In [14]:
import os
import subprocess

# PYTHON_COLORS=0 disables the subprocess's ANSI-colored traceback so the captured
# stderr text below is plain and paste-able.
clean_env = dict(os.environ, PYTHON_COLORS="0")

os.makedirs("/tmp/circ_demo", exist_ok=True)
with open("/tmp/circ_demo/circ_a.py", "w") as f:
    f.write("from circ_b import Y\nX = 1\n")
with open("/tmp/circ_demo/circ_b.py", "w") as f:
    f.write("from circ_a import X\nY = 2\n")

result = subprocess.run(
    [sys.executable, "-c", "import circ_a"],
    cwd="/tmp/circ_demo", capture_output=True, text=True, env=clean_env,
)
print("returncode:", result.returncode)
print("last stderr line:", result.stderr.strip().splitlines()[-1])

returncode: 1
last stderr line: ImportError: cannot import name 'X' from 'circ_a' (consider renaming '/tmp/circ_demo/circ_a.py' if it has the same name as a library you intended to import)


**Shadowing a standard-library module name.** `sys.path`'s search order means a local file
literally named `random.py` sitting in (or ahead of) a searched directory is found *before* the
real standard-library `random` — `import random` then binds to the local file, silently, with no
error at the import line itself. The failure only surfaces later, wherever the real module's
functionality is actually used.

In [15]:
os.makedirs("/tmp/shadow_demo", exist_ok=True)
with open("/tmp/shadow_demo/random.py", "w") as f:
    f.write("# a project file that happens to be named random.py\nMY_CONSTANT = 7\n")

shadowed = subprocess.run(
    [sys.executable, "-c", "import random; print(random.randint(1, 10))"],
    cwd="/tmp/shadow_demo", capture_output=True, text=True, env=clean_env,
)
print("--- run from inside shadow_demo/ (local random.py shadows stdlib) ---")
print("returncode:", shadowed.returncode)
print("last stderr line:", shadowed.stderr.strip().splitlines()[-1])

control = subprocess.run(
    [sys.executable, "-c", "import random; print(random.randint(1, 10))"],
    cwd="/tmp", capture_output=True, text=True, env=clean_env,
)
print("--- run from /tmp (no shadowing file present) ---")
print("returncode:", control.returncode)
print("stdout:", control.stdout.strip())

--- run from inside shadow_demo/ (local random.py shadows stdlib) ---
returncode: 1
last stderr line: AttributeError: module 'random' has no attribute 'randint' (consider renaming '/tmp/shadow_demo/random.py' since it has the same name as the standard library module named 'random' and prevents importing that standard library module)
--- run from /tmp (no shadowing file present) ---
returncode: 0
stdout: 5


## 9. Installing Packages with `uv`

All packages in this repo are managed with `uv`. Run from repo root:

```bash
# Install all dependencies
uv sync

# Add a new package
uv add numpy

# Add a dev-only package
uv add --dev pytest

# Run a script with the venv
uv run script.py
```

Check installed packages:
```bash
uv pip list
```

## Quick Summary

| Tool | Purpose |
|------|---------|
| `import module` | Import a whole module |
| `from module import x` | Import specific name |
| `__name__ == '__main__'` | Guard for script-only code |
| `pathlib.Path` | Modern cross-platform file paths |
| `json` | Serialise Python ↔ JSON |
| `re` | Pattern matching on strings |
| `collections.Counter` | Word/item frequency counting |
| `itertools` | Efficient iteration recipes |

**Next →** [06 – File Handling & Exceptions](../06-file-exception/)